# Kalman Filter Use Cases — TFiltersPy

This notebook demonstrates the sklearn-compatible `KalmanFilter` API from TFiltersPy on three
real-world use cases:

1. **Image Denoising** (Computer Vision) — denoise handwritten digit images
2. **Multivariate EEG Time Series** — smooth 14-channel EEG recordings
3. **NLP Topic Smoothing** — smooth LDA topic distributions from disaster tweets

Each section uses the same core workflow:
```python
kf = KalmanFilter(F, H, Q, R, x0, P0)
kf.fit(measurements)
states = kf.predict()           # filtered estimates
smoothed, covs = kf.smooth()    # RTS smoother
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tfilterspy import KalmanFilter, DaskKalmanFilter

---

## 1. Image Denoising (Computer Vision)

We treat each 8x8 digit image (64 pixels) as a measurement vector in a static state-space model.
The Kalman filter estimates the clean pixel intensities from noisy observations.

- **State transition:** Identity (pixels don't change between samples)
- **Observation model:** Identity (we observe all pixels directly)
- **Process noise Q:** Small (the true image is nearly static)
- **Observation noise R:** Larger (captures the Gaussian noise we added)

In [ ]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

# Load digits (2000 samples, 64-dimensional)
digits = load_digits()
X_digits = digits.data[:2000]
y_digits = digits.target[:2000]

X_train, X_test, y_train, y_test = train_test_split(
    X_digits, y_digits, test_size=0.2, random_state=42
)

# Add Gaussian noise
noise_level = 0.88
rng = np.random.RandomState(42)
noisy_train = X_train + rng.normal(0, noise_level, X_train.shape)
noisy_test = X_test + rng.normal(0, noise_level, X_test.shape)

print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")
print(f"Noise level: {noise_level}")

In [ ]:
# Build Kalman Filter for 64-dimensional pixel space
n = 64
F = np.eye(n)
H = np.eye(n)
Q = 0.01 * np.eye(n)
R = 0.1 * np.eye(n)
x0 = np.zeros(n)
P0 = np.eye(n)

# --- Filter training data ---
kf_train = KalmanFilter(F, H, Q, R, x0, P0)
kf_train.fit(noisy_train)
filtered_train = kf_train.predict()
smoothed_train, smoothed_train_covs = kf_train.smooth()

# --- Filter test data ---
kf_test = KalmanFilter(F, H, Q, R, x0, P0)
kf_test.fit(noisy_test)
filtered_test = kf_test.predict()
smoothed_test, smoothed_test_covs = kf_test.smooth()

print(f"Filtered train shape: {filtered_train.shape}")
print(f"Smoothed train shape: {smoothed_train.shape}")

In [ ]:
# Visualize: Original / Noisy / Filtered / Smoothed
sample_idx = 0

images = [
    X_test[sample_idx].reshape(8, 8),
    noisy_test[sample_idx].reshape(8, 8),
    filtered_test[sample_idx].reshape(8, 8),
    smoothed_test[sample_idx].reshape(8, 8),
]
titles = ["Original", "Noisy", "Filtered", "Smoothed"]

fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
for ax, img, title in zip(axes, images, titles):
    ax.imshow(img, cmap="gray", interpolation="nearest")
    ax.set_title(title)
    ax.axis("off")
plt.suptitle(f"Digit = {y_test[sample_idx]}", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Classification accuracy comparison
clf = LogisticRegression(max_iter=5000, random_state=42)

# On noisy data
clf.fit(noisy_train, y_train)
acc_noisy = clf.score(noisy_test, y_test)

# On filtered data
clf.fit(filtered_train, y_train)
acc_filtered = clf.score(filtered_test, y_test)

# On smoothed data
clf.fit(smoothed_train, y_train)
acc_smoothed = clf.score(smoothed_test, y_test)

print(f"Classification accuracy (noisy):    {acc_noisy:.4f}")
print(f"Classification accuracy (filtered): {acc_filtered:.4f}")
print(f"Classification accuracy (smoothed): {acc_smoothed:.4f}")

# MSE comparison
mse_noisy    = np.mean((noisy_test - X_test) ** 2)
mse_filtered = np.mean((filtered_test - X_test) ** 2)
mse_smoothed = np.mean((smoothed_test - X_test) ** 2)

print(f"\nMSE (noisy vs original):    {mse_noisy:.4f}")
print(f"MSE (filtered vs original): {mse_filtered:.4f}")
print(f"MSE (smoothed vs original): {mse_smoothed:.4f}")

---

## 2. Multivariate EEG Time Series

The EEG Eye State dataset has 14 channels recorded over ~15,000 timesteps. We use the
Kalman filter to smooth the raw signals, treating each timestep as a measurement of the
14-dimensional state.

Both `predict()` (forward filter) and `smooth()` (RTS smoother) are compared.

In [ ]:
from sklearn.datasets import fetch_openml

# Load EEG data (first 9000 rows)
eeg_data = fetch_openml('eeg-eye-state', version=1, as_frame=False, parser='liac-arff')
X_eeg = eeg_data.data[:9000].astype(np.float64)
print(f"EEG data shape: {X_eeg.shape}  (timesteps x channels)")

In [ ]:
# Build Kalman Filter for 14-channel EEG
n_ch = 14
F_eeg = np.eye(n_ch)
H_eeg = np.eye(n_ch)
Q_eeg = 0.01 * np.eye(n_ch)
R_eeg = 0.1 * np.eye(n_ch)
x0_eeg = np.zeros(n_ch)
P0_eeg = np.eye(n_ch)

kf_eeg = KalmanFilter(F_eeg, H_eeg, Q_eeg, R_eeg, x0_eeg, P0_eeg)
kf_eeg.fit(X_eeg)

filtered_eeg = kf_eeg.predict()
smoothed_eeg, smoothed_eeg_covs = kf_eeg.smooth()

# MSE
mse_filtered_eeg = np.mean((X_eeg - filtered_eeg) ** 2)
mse_smoothed_eeg = np.mean((X_eeg - smoothed_eeg) ** 2)
print(f"MSE (raw vs filtered): {mse_filtered_eeg:.4f}")
print(f"MSE (raw vs smoothed): {mse_smoothed_eeg:.4f}")

In [ ]:
# Plot Channel 1: Raw vs Filtered vs Smoothed (first 1000 timesteps)
t = 1000
ch = 0

plt.figure(figsize=(14, 4))
plt.plot(X_eeg[:t, ch], label="Raw", alpha=0.5)
plt.plot(filtered_eeg[:t, ch], label="Filtered", linestyle="--")
plt.plot(smoothed_eeg[:t, ch], label="Smoothed", linestyle=":")
plt.title("EEG Channel 1 -- Raw vs Filtered vs Smoothed")
plt.xlabel("Timestep")
plt.ylabel("Amplitude")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# All 14 channels (7x2 subplots, first 1000 timesteps)
fig, axes = plt.subplots(7, 2, figsize=(16, 18), sharex=True)
for i, ax in enumerate(axes.flat):
    ax.plot(X_eeg[:t, i], label="Raw", alpha=0.4)
    ax.plot(filtered_eeg[:t, i], label="Filtered", linestyle="--", alpha=0.8)
    ax.plot(smoothed_eeg[:t, i], label="Smoothed", linestyle=":", alpha=0.8)
    ax.set_title(f"Channel {i + 1}", fontsize=10)
    if i == 0:
        ax.legend(fontsize=8)
fig.suptitle("All 14 EEG Channels (first 1000 timesteps)", fontsize=14)
plt.tight_layout()
plt.show()

---

## 3. NLP Topic Smoothing (Disaster Tweets)

We extract 5 topics from disaster tweets using LDA, then treat the sequence of topic
distributions as noisy measurements in a 5-dimensional state space. The Kalman filter
smooths out the noisy per-tweet topic probabilities, revealing cleaner temporal trends.

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

# Load disaster tweets
data_path = '../data/train_nlp.csv'
df = pd.read_csv(data_path)
tweets = df['text'].values[:5000]
print(f"Number of tweets: {len(tweets)}")

# Extract 5 topics with LDA
vectorizer = CountVectorizer(max_features=5000, stop_words='english')
X_bow = vectorizer.fit_transform(tweets)

n_topics = 5
lda = LatentDirichletAllocation(n_components=n_topics, random_state=42)
topic_dist = lda.fit_transform(X_bow)  # shape: (n_tweets, 5)
print(f"Topic distribution shape: {topic_dist.shape}")

In [ ]:
# Build Kalman Filter for topic smoothing
F_nlp = np.eye(n_topics)
H_nlp = np.eye(n_topics)
Q_nlp = 0.01 * np.eye(n_topics)
R_nlp = 0.1 * np.eye(n_topics)
x0_nlp = topic_dist[0]  # initialise with the first topic vector
P0_nlp = np.eye(n_topics)

kf_nlp = KalmanFilter(F_nlp, H_nlp, Q_nlp, R_nlp, x0_nlp, P0_nlp)
kf_nlp.fit(topic_dist)
smoothed_topics = kf_nlp.predict()

print(f"Smoothed topics shape: {smoothed_topics.shape}")

In [ ]:
# Plot raw vs smoothed topics (5 subplots)
fig, axes = plt.subplots(n_topics, 1, figsize=(14, 12), sharex=True)
for i, ax in enumerate(axes):
    ax.plot(topic_dist[:, i], label="Raw", alpha=0.4)
    ax.plot(smoothed_topics[:, i], label="Smoothed", linestyle="--")
    ax.set_title(f"Topic {i + 1}")
    ax.set_ylabel("Probability")
    ax.legend(fontsize=8)
axes[-1].set_xlabel("Tweet Index")
fig.suptitle("LDA Topic Distributions -- Raw vs Kalman-Smoothed", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Top words per topic
feature_names = vectorizer.get_feature_names_out()
for i, component in enumerate(lda.components_):
    top_words = [feature_names[j] for j in component.argsort()[-10:][::-1]]
    print(f"Topic {i + 1}: {', '.join(top_words)}")